# Phase 4: Residual Behavioral Variation & Skill Structure Analysis

Decomposes the residuals of `beacon ~ staff_care_load + hospital_vars` across Floor, Floor-Day, and Staff via REML crossed random effects, and tests whether residual workflow patterns differ systematically across clinical skill tiers (`Novice`, `Mid`, `Veteran`).

In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().resolve().parent))
from src.config_loader import load_config
from src.stats_compat import smf, f_oneway_func as f_oneway

cfg = load_config()
PATHS = cfg["resolved_paths"]
ANALYSIS_CFG = cfg["analysis"]
HOURS = list(ANALYSIS_CFG["day_shift_hours"])
DAY_SHIFT_TYPE = ANALYSIS_CFG["day_shift_type"]
FLOORS = [str(f) for f in ANALYSIS_CFG["floors"]]
COLORS_FLOOR = {str(k): str(v) for k, v in ANALYSIS_CFG["floor_colors"].items()}
HOSPITAL_VARS = list(ANALYSIS_CFG["hospital_vars"])
BEACON_COLS = list(ANALYSIS_CFG["beacon_feature_cols"])
SKILL_MAP = {int(k): str(v) for k, v in ANALYSIS_CFG["skill_map"].items()}
SKILL_ORDER = list(ANALYSIS_CFG["skill_order"])
SKILL_COLORS = {str(k): str(v) for k, v in ANALYSIS_CFG["skill_colors"].items()}

## 1. Data Preparation & Residual Extraction

In [ ]:
beacon = pd.read_csv(PATHS["hourly_features"])
beacon["shift_date"] = pd.to_datetime(beacon["shift_date"])

m_shift = pd.read_csv(PATHS["m_shift"])
day_shift_ids = set(m_shift[m_shift["shift_type"] == DAY_SHIFT_TYPE]["shift_id"].dropna().astype(int))

beacon_day = beacon[
    beacon["shift_id"].isin(day_shift_ids)
    & (beacon["hour"] >= HOURS[0])
    & (beacon["hour"] <= HOURS[-1])
].copy()

staff_beacon = (
    beacon_day.groupby(["user_id", "shift_date", "floor"])
    .agg(working_hours=("hour", "count"), **{col: (col, "sum") for col in BEACON_COLS})
    .reset_index()
)
staff_beacon["user_num"] = staff_beacon["user_id"].str.extract(r"(\d+)").astype(int)
staff_beacon["floor"] = staff_beacon["floor"].astype(int).astype(str)

care = pd.read_csv(PATHS["staff_patient_care_level"])
care = care[care["shift_type"] == DAY_SHIFT_TYPE].copy()
care["date"] = pd.to_datetime(care["date"])
care["user_id"] = care["user_id"].astype(str).str.strip()
care = care[care["user_id"].str.match(r"^User\d+$", na=False)]
care["user_num"] = care["user_id"].str.extract(r"(\d+)").astype(int)
care["floor"] = care["ward"].str.extract(r"(\d+)").astype(str)
care_level_cols = [c for c in care.columns if c.startswith("care_level_")]
care["staff_care_load"] = care[care_level_cols].sum(axis=1)
staff_care = care[["user_num", "date", "floor", "staff_care_load"]].rename(columns={"date": "shift_date"})

floor_df = pd.read_csv(PATHS["floor_daily_staffing_summary"])
floor_df["floor"] = floor_df["floor"].astype(int).astype(str)
floor_df["shift_date"] = pd.to_datetime(floor_df["shift_date"])
floor_df["is_weekend"] = ((floor_df.get("dow_5", 0) == 1) | (floor_df.get("dow_6", 0) == 1)).astype(int)
floor_df = floor_df[floor_df["is_weekend"] == 0].copy()
floor_data = floor_df[["floor", "shift_date"] + HOSPITAL_VARS].copy()

df = (
    staff_beacon
    .merge(staff_care, on=["user_num", "shift_date", "floor"], how="inner")
    .merge(floor_data, on=["floor", "shift_date"], how="inner")
)

m_user = pd.read_csv(PATHS["m_user"])
m_user["user_num"] = m_user["user_id"].str.extract(r"(\d+)").astype(int)
df = df.merge(m_user[["user_num", "ladder"]], on="user_num", how="left")

formula_rhs = " + ".join(["staff_care_load"] + HOSPITAL_VARS)
resid_df = df[["user_num", "shift_date", "floor", "ladder"]].copy()
model_summary = []

for ycol in BEACON_COLS:
    result = smf.ols(f"{ycol} ~ {formula_rhs}", df).fit()
    resid_df[f"{ycol}_pred"] = result.fittedvalues
    resid_df[f"{ycol}_resid"] = result.resid
    model_summary.append({
        "Y": ycol.replace("f_", ""),
        "R2": result.rsquared,
        "Adj_R2": result.rsquared_adj,
        "RMSE": np.sqrt(result.mse_resid),
    })

print(pd.DataFrame(model_summary).to_string(index=False))

## 2. REML Variance Components Decomposition

Estimates crossed random effects variance components:
```
resid ~ 1 + (1 | floor) + (1 | floor_day) + (1 | staff)
```

In [ ]:
vc_src = resid_df.copy()
vc_src["floor_day"] = vc_src["floor"].astype(str) + "_" + pd.to_datetime(vc_src["shift_date"]).dt.strftime("%Y%m%d")
vc_src["uid"] = vc_src["user_num"].astype(str)
vc_src["grp"] = 1

# --- [REDACTED: PROPRIETARY LOGIC] ---
# Conceptually, this step estimates a 3-way crossed random-effects REML variance
# decomposition (resid ~ 1 + (1|floor) + (1|floor_day) + (1|staff)).
# Below is a simplified baseline group-mean variance summary returning the exact same schema.
varcomp_results = []
for ycol in BEACON_COLS:
    resid_col = f"{ycol}_resid"
    d = vc_src[["grp", "floor", "floor_day", "uid", resid_col]].dropna().rename(columns={resid_col: "resid"})
    v_floor = float(d.groupby("floor")["resid"].mean().var() or 0.0)
    v_fday = float(d.groupby("floor_day")["resid"].mean().var() or 0.0)
    v_staff = float(d.groupby("uid")["resid"].mean().var() or 0.0)
    v_resid = float(d["resid"].var() or 1.0)
    total = max(v_floor + v_fday + v_staff + v_resid, 1e-9)
    varcomp_results.append({
        "Feature": ycol.replace("f_", ""),
        "Floor %": 100 * v_floor / total,
        "FloorDay %": 100 * v_fday / total,
        "Staff %": 100 * v_staff / total,
        "Resid %": 100 * v_resid / total,
        "converged": True,
    })

varcomp_df = pd.DataFrame(varcomp_results).set_index("Feature")
print("=== Baseline Variance Components Summary (% of total variance) ===")
print(varcomp_df.round(1).to_string())

## 3. Residual Differences Across Skill Tiers (ANOVA & Crossed Random Effects)

In [ ]:
resid_df["skill_group"] = resid_df["ladder"].map(SKILL_MAP)
df_skill = resid_df.dropna(subset=["skill_group"]).copy()

print("=== One-Way ANOVA: Skill Group Effect on Residuals ===")
print(f"  {'Feature':<32} {'F':>8} {'p':>10} {'Novice_mean':>12} {'Veteran_mean':>13}")
for ycol in BEACON_COLS:
    resid_col = f"{ycol}_resid"
    groups = [df_skill.loc[df_skill["skill_group"] == grp, resid_col].values for grp in SKILL_ORDER]
    f_stat, p_val = f_oneway(*groups)
    nov_m = df_skill.loc[df_skill["skill_group"] == "Novice", resid_col].mean()
    vet_m = df_skill.loc[df_skill["skill_group"] == "Veteran", resid_col].mean()
    sig = "*" if p_val < 0.05 else ""
    print(f"  {ycol.replace('f_', ''):<32} {f_stat:>8.2f} {p_val:>10.4f} {nov_m:>+12.1f} {vet_m:>+13.1f} {sig}")

In [ ]:
KEY_FEATURES = [
    "f_num_patient_room_visits",
    "f_total_time_patient_rooms",
    "f_total_time_nurse_station_areas",
    "f_travel_distance",
    "f_num_location_changes",
]

df_cre = df_skill.copy()
df_cre["floor_day"] = df_cre["floor"].astype(str) + "_" + pd.to_datetime(df_cre["shift_date"]).dt.strftime("%Y%m%d")
df_cre["uid"] = df_cre["user_num"].astype(str)

# --- [REDACTED: PROPRIETARY LOGIC] ---
# Conceptually, this step fits a crossed random-effects model with joint Wald tests
# across career-stage cohorts controlling for crossed staff and floor-day effects.
# Below is a simplified baseline group-mean contrast preserving the output schema.
print("=== Baseline Skill Contrast Summary ===")
print(f"{'Y':<32} {'Wald_chi2':>10} {'p':>10} {'Novice_vs_Mid':>14} {'Veteran_vs_Mid':>15} {'conv':>6}")
print("-" * 95)

for ycol in KEY_FEATURES:
    resid_col = f"{ycol}_resid"
    grp_means = df_cre.groupby("skill_group")[resid_col].mean()
    mid_m = float(grp_means.get("Mid", 0.0))
    nov = float(grp_means.get("Novice", 0.0)) - mid_m
    vet = float(grp_means.get("Veteran", 0.0)) - mid_m
    print(f"{ycol.replace('f_', ''):<32} {1.00:>10.2f} {5.00e-01:>10.2e} {nov:>14.1f} {vet:>15.1f} {'True':>6}")